In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Settings
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Libraries imported successfully!")

## 1. Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv('../../alzheimers_disease_data.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nTarget Distribution:")
print(df['Diagnosis'].value_counts())
print(f"\nClass Balance: {(df['Diagnosis'].value_counts(normalize=True) * 100).round(2).to_dict()}")

# Display first few rows
df.head()

## 2. Feature Selection

Using the same 32 features as other models (excluding PatientID).

In [ ]:
# Feature columns (32 features)
FEATURE_COLUMNS = [
    'Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI', 'Smoking',
    'AlcoholConsumption', 'PhysicalActivity', 'DietQuality', 'SleepQuality',
    'FamilyHistoryAlzheimers', 'CardiovascularDisease', 'Diabetes',
    'Depression', 'HeadInjury', 'Hypertension', 'SystolicBP', 'DiastolicBP',
    'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL', 'CholesterolTriglycerides',
    'MMSE', 'FunctionalAssessment', 'MemoryComplaints', 'BehavioralProblems',
    'ADL', 'Confusion', 'Disorientation', 'PersonalityChanges',
    'DifficultyCompletingTasks', 'Forgetfulness'
]

TARGET = 'Diagnosis'

# Prepare features and target
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET].copy()

print(f"Features: {len(FEATURE_COLUMNS)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nNo missing values: {X.isnull().sum().sum() == 0}")

## 3. Train-Test Split

In [ ]:
# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

## 4. Hyperparameter Tuning with GridSearchCV

Training Random Forest with cross-validation to find optimal parameters.

In [ ]:
# Parameter grid for Random Forest
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Create model
rf_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

# GridSearchCV
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

print("Training Random Forest with GridSearchCV...")
print("This may take a few minutes...\n")

grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

print(f"\n✓ Random Forest training completed!")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

## 5. Model Evaluation

In [ ]:
# Predictions
y_pred = best_rf.predict(X_test)
y_pred_proba = best_rf.predict_proba(X_test)[:, 1]
y_pred_train = best_rf.predict(X_train)

# Calculate metrics
rf_test_acc = accuracy_score(y_test, y_pred)
rf_train_acc = accuracy_score(y_train, y_pred_train)
rf_precision = precision_score(y_test, y_pred)
rf_recall = recall_score(y_test, y_pred)
rf_f1 = f1_score(y_test, y_pred)
rf_auc = roc_auc_score(y_test, y_pred_proba)

# Cross-validation on full dataset
rf_cv_scores = cross_val_score(best_rf, X, y, cv=10, scoring='accuracy')

print("="*70)
print("RANDOM FOREST PERFORMANCE")
print("="*70)
print(f"Test Accuracy:  {rf_test_acc:.4f}")
print(f"Train Accuracy: {rf_train_acc:.4f}")
print(f"Precision:      {rf_precision:.4f}")
print(f"Recall:         {rf_recall:.4f}")
print(f"F1-Score:       {rf_f1:.4f}")
print(f"ROC-AUC:        {rf_auc:.4f}")
print(f"CV Accuracy:    {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std() * 2:.4f})")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Diagnosis', 'Diagnosis']))

## 6. Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix - Random Forest', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks([0.5, 1.5], ['No Diagnosis', 'Diagnosis'])
plt.yticks([0.5, 1.5], ['No Diagnosis', 'Diagnosis'])
plt.tight_layout()
plt.show()

print("Confusion Matrix:")
print(cm)

## 7. ROC Curve

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {rf_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Random Forest', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Feature Importance Analysis

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("="*70)
print("TOP 15 MOST IMPORTANT FEATURES")
print("="*70)
print(importance_df.head(15).to_string(index=False))

# Plot feature importance
plt.figure(figsize=(10, 8))
top_features = importance_df.head(15)
plt.barh(range(len(top_features)), top_features['importance'], color='steelblue')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance', fontsize=12)
plt.title('Top 15 Feature Importance - Random Forest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Save Model and Artifacts

In [ ]:
# Save model
joblib.dump(best_rf, 'random_forest_model.pkl')
print(f"✓ Model saved: random_forest_model.pkl")

# Save metrics
metrics = {
    'model': 'Random Forest',
    'accuracy': float(rf_test_acc),
    'train_accuracy': float(rf_train_acc),
    'precision': float(rf_precision),
    'recall': float(rf_recall),
    'f1_score': float(rf_f1),
    'roc_auc': float(rf_auc),
    'cv_mean': float(rf_cv_scores.mean()),
    'cv_std': float(rf_cv_scores.std()),
    'best_params': grid_search.best_params_,
    'n_estimators': int(best_rf.n_estimators),
    'n_features': len(FEATURE_COLUMNS)
}

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)
print(f"✓ Metrics saved: metrics.json")

# Save predictions
predictions_df = pd.DataFrame({
    'true_label': y_test.values,
    'predicted_label': y_pred,
    'probability_no_diagnosis': best_rf.predict_proba(X_test)[:, 0],
    'probability_diagnosis': best_rf.predict_proba(X_test)[:, 1]
})
predictions_df.to_csv('predictions.csv', index=False)
print(f"✓ Predictions saved: predictions.csv")

# Save feature importance
importance_df.to_csv('feature_importance.csv', index=False)
print(f"✓ Feature importance saved: feature_importance.csv")

# Save feature names
joblib.dump(FEATURE_COLUMNS, 'feature_names.pkl')
print(f"✓ Feature names saved: feature_names.pkl")

print("\n" + "="*70)
print("✅ RANDOM FOREST TRAINING COMPLETE!")
print("="*70)